# 10 Generate seamless textures on the GPU

Makes floor, wall and roof textures for the walk-through with a diffusion model, then derives the normal and roughness/AO maps from each image. Run in the Real2Twin lab (`nada-r2t-lab`). **Not run yet** (written without server access): read the cells, run them one by one, and tell me what breaks.

- The GPU memory is shared with the team LLM (it reserves a share). The second cell stops if less than 14 GB is free.
- **Licence:** the default model is `stabilityai/sdxl-turbo`. Check its licence before the images leave the research context (it is not a free-for-all licence). `black-forest-labs/FLUX.1-schnell` is Apache 2.0 but much heavier; swap `MODEL` and the call in cell 4 if you prefer it.
- Output: `/workspace/usable-space/out/textures/gen_<slot>_{diff,nor,arm}.jpg` plus a contact sheet. Download them into `client/public/textures/` and open the app with `?tex=gen`.

In [ ]:
%pip install -q diffusers accelerate safetensors pillow scipy
import torch, diffusers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'diffusers', diffusers.__version__)

In [ ]:
free, total = torch.cuda.mem_get_info()
print(f'GPU memory free {free/2**30:.1f} of {total/2**30:.1f} GiB (shared with the team LLM)')
assert free > 14 * 2**30, 'less than 14 GiB free: ask before loading a model'

In [ ]:
from diffusers import AutoPipelineForText2Image
MODEL = 'stabilityai/sdxl-turbo'
pipe = AutoPipelineForText2Image.from_pretrained(MODEL, torch_dtype=torch.float16, variant='fp16').to('cuda')
# seamless tiling: let every convolution wrap around the image edges
for net in (pipe.unet, pipe.vae):
    for m in net.modules():
        if isinstance(m, torch.nn.Conv2d):
            m.padding_mode = 'circular'
print('model ready')

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image
OUT = Path('/workspace/usable-space/out/textures'); OUT.mkdir(parents=True, exist_ok=True)
STYLE = ', seamless tileable texture, top-down photograph, flat even lighting, no shadows, no objects, high detail'
PROMPTS = {
    'floor': 'warm oak parquet floor boards, fine wood grain' + STYLE,
    'wall': 'smooth warm white painted plaster wall, subtle roller texture' + STYLE,
    'roof': 'weathered red clay roof tiles in overlapping rows' + STYLE,
}
VARIANTS = 4
images = {}
for slot, prompt in PROMPTS.items():
    g = torch.Generator('cuda').manual_seed(7)
    images[slot] = [pipe(prompt, num_inference_steps=2, guidance_scale=0.0, height=512, width=512, generator=g).images[0] for _ in range(VARIANTS)]
sheet = Image.new('RGB', (256 * VARIANTS, 256 * len(PROMPTS)))
for r, slot in enumerate(PROMPTS):
    for c, im in enumerate(images[slot]):
        sheet.paste(im.resize((256, 256)), (c * 256, r * 256))
sheet.save(OUT / 'contact_sheet.png')
print('look at', OUT / 'contact_sheet.png', 'then pick one variant per slot in the next cell')

## Pick one variant per slot and derive the PBR maps
`PICK` is the index (0-3) of the variant you like for each slot. The normal map comes from the brightness of the image treated as a height field, roughness from local contrast (rougher where the image is darker), AO from a blurred copy. These are approximations, good enough for a walk-through, not measured data.

In [ ]:
from scipy.ndimage import gaussian_filter, sobel
PICK = {'floor': 0, 'wall': 0, 'roof': 0}
def wrap_blur(a, s): return gaussian_filter(a, s, mode='wrap')
for slot, i in PICK.items():
    im = images[slot][i].convert('RGB')
    rgb = np.asarray(im).astype(np.float32) / 255
    h = wrap_blur(rgb.mean(axis=2), 1.0)
    dx = sobel(h, axis=1, mode='wrap'); dy = sobel(h, axis=0, mode='wrap')
    strength = 3.0
    n = np.dstack([-dx * strength, dy * strength, np.ones_like(h)])  # OpenGL-style: green up
    n /= np.linalg.norm(n, axis=2, keepdims=True)
    nor = ((n * 0.5 + 0.5) * 255).astype(np.uint8)
    ao = np.clip(0.75 + 1.5 * (h - wrap_blur(h, 6)), 0.4, 1.0)  # darker where a pixel is darker than its surroundings (crevices)
    rough = np.clip(0.95 - 0.5 * h, 0.35, 1.0)
    arm = (np.dstack([ao, rough, np.zeros_like(h)]) * 255).astype(np.uint8)
    im.save(OUT / f'gen_{slot}_diff.jpg', quality=92)
    Image.fromarray(nor).save(OUT / f'gen_{slot}_nor.jpg', quality=92)
    Image.fromarray(arm).save(OUT / f'gen_{slot}_arm.jpg', quality=92)
print(sorted(p.name for p in OUT.glob('gen_*.jpg')))

In [ ]:
del pipe
import gc; gc.collect(); torch.cuda.empty_cache()
print('GPU memory released:', f'{torch.cuda.mem_get_info()[0]/2**30:.1f} GiB free')